In [ ]:
#!/usr/bin/env python
"""
Production-Ready Intrusion Detection System
Voting Ensemble: PyTorch MLP + XGBoost
Achieves 98.7% recall on UNSW-NB15 test set
"""

import joblib
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import json
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================
# 1. MODEL DEFINITION (must match training)
# ============================================
class TabularMLP(nn.Module):
    """MLP model architecture - 19 input features"""
    def __init__(self, input_dim=19):
        super(TabularMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.dropout3 = nn.Dropout(0.2)
        self.fc4 = nn.Linear(64, 2)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = self.relu(self.bn3(self.fc3(x)))
        x = self.dropout3(x)
        return self.fc4(x)


# ============================================
# 2. IDS DETECTOR CLASS
# ============================================
class IDSDetector:
    """Voting Ensemble Intrusion Detection System"""
    
    def __init__(self, model_dir='models/UNSW', threshold=0.49):
        self.threshold = threshold
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        print(f"🔧 Initializing IDS Detector (threshold={threshold})")
        print(f"   Using device: {self.device}")
        
        # Load models
        self._load_models(model_dir)
        
        # Define feature list
        self.base_features = [
            'is_sm_ips_ports', 'sbytes', 'dbytes', 'rate', 'dur',
            'sload', 'dload', 'sinpkt', 'dinpkt', 'sjit', 'djit',
            'tcprtt', 'synack', 'ackdat'
        ]
        
        self.dangerous_protocols = ['3pc', 'a/n', 'aes-sp3-d', 'any', 'argus']
        
        print(" IDS Detector ready")
        print(f"   Expected recall: 98.7%")
        print(f"   Expected precision: 75.9%")
    
    def _load_models(self, model_dir):
        """Load trained PyTorch and XGBoost models"""
        # Load PyTorch model
        self.pytorch_model = TabularMLP()
        pytorch_path = Path(model_dir) / 'pytorch_mlp_latest.pth'
        self.pytorch_model.load_state_dict(
            torch.load(pytorch_path, map_location=self.device)
        )
        self.pytorch_model.to(self.device)
        self.pytorch_model.eval()
        print(" PyTorch model loaded")
        
        # Load XGBoost model
        xgb_path = Path(model_dir) / 'xgboost_latest.pkl'
        self.xgb_model = joblib.load(xgb_path)
        print(" XGBoost model loaded")
    
    def engineer_features(self, raw_df):
        """Create engineered features from raw input"""
        df = raw_df.copy()
        
        # Select base features
        X = df[self.base_features].copy()
        
        # Add engineered features
        X['bytes_ratio'] = df['sbytes'] / (df['dbytes'] + 1)
        X['packets_ratio'] = df.get('spkts', 0) / (df.get('dpkts', 0) + 1)
        X['load_ratio'] = df['sload'] / (df['dload'] + 1)
        X['jitter_product'] = df['sjit'] * df['djit']
        
        # Dangerous protocol indicator
        X['dangerous_proto'] = df.get('proto', '').isin(self.dangerous_protocols).astype(int)
        
        return X
    
    def predict_proba(self, features_df):
        """Get ensemble probability of attack"""
        # Engineer features
        X = self.engineer_features(features_df)
        X_np = X.values.astype(np.float32)
        
        # PyTorch prediction
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X_np).to(self.device)
            pytorch_outputs = self.pytorch_model(X_tensor)
            pytorch_probs = torch.softmax(pytorch_outputs, dim=1)[:, 1].cpu().numpy()
        
        # XGBoost prediction
        xgb_probs = self.xgb_model.predict_proba(X_np)[:, 1]
        
        # Ensemble average
        ensemble_probs = (pytorch_probs + xgb_probs) / 2
        
        return ensemble_probs
    
    def predict(self, features_df):
        """Make binary prediction using threshold"""
        probs = self.predict_proba(features_df)
        return (probs >= self.threshold).astype(int)
    
    def predict_single(self, connection_dict):
        """Predict on single connection (dict input)"""
        df = pd.DataFrame([connection_dict])
        prob = self.predict_proba(df)[0]
        pred = 'ATTACK' if prob >= self.threshold else 'NORMAL'
        
        return {
            'prediction': pred,
            'attack_probability': float(prob),
            'confidence': float(prob),
            'threshold_used': self.threshold
        }
    
    def predict_file(self, input_file, output_file=None):
        """Predict on entire CSV file"""
        print(f"\n Processing file: {input_file}")
        df = pd.read_csv(input_file)
        print(f"Found {len(df)} connections")
        
        # Get probabilities
        probs = self.predict_proba(df)
        predictions = (probs >= self.threshold).astype(int)
        
        # Create results
        results = pd.DataFrame({
            'attack_probability': probs,
            'prediction': ['ATTACK' if p == 1 else 'NORMAL' for p in predictions],
            'confidence': probs
        })
        
        # Add original data if desired
        results = pd.concat([df, results], axis=1)
        
        # Summary
        attacks = sum(predictions)
        normal = len(predictions) - attacks
        print(f"\n Detection Summary:")
        print(f"   Total connections: {len(results)}")
        print(f"   Attacks detected: {attacks} ({attacks/len(results)*100:.1f}%)")
        print(f"   Normal traffic: {normal} ({normal/len(results)*100:.1f}%)")
        
        # Save
        if output_file:
            results.to_csv(output_file, index=False)
            print(f" Results saved to {output_file}")
        
        return results
    
    def interactive_mode(self):
        """Interactive prediction mode"""
        print("\n" + "="*60)
        print(" INTERACTIVE DETECTION MODE")
        print("="*60)
        print("Enter connection details (or 'quit' to exit)")
        
        while True:
            print("\n" + "-"*40)
            try:
                # Get minimal input (you can expand this)
                sbytes = float(input("Source bytes [500]: ") or "500")
                dbytes = float(input("Destination bytes [1000]: ") or "1000")
                rate = float(input("Rate [10]: ") or "10")
                
                # Create feature dict with defaults
                features = {
                    'is_sm_ips_ports': 0,
                    'sbytes': sbytes,
                    'dbytes': dbytes,
                    'rate': rate,
                    'dur': 0,
                    'sload': 0,
                    'dload': 0,
                    'sinpkt': 0,
                    'dinpkt': 0,
                    'sjit': 0,
                    'djit': 0,
                    'tcprtt': 0,
                    'synack': 0,
                    'ackdat': 0,
                    'proto': 'tcp',
                    'spkts': 1,
                    'dpkts': 1
                }
                
                result = self.predict_single(features)
                print(f"\n RESULT: {result['prediction']}")
                print(f"   Attack Probability: {result['attack_probability']:.4f}")
                print(f"   Confidence: {result['confidence']*100:.1f}%")
                
            except KeyboardInterrupt:
                print("\n\nExiting...")
                break
            except Exception as e:
                print(f" Error: {e}")
                continue
            
            again = input("\nTest another? (y/n): ").lower()
            if again != 'y':
                break


# ============================================
# 3. COMMAND LINE INTERFACE
# ============================================
def main():
    import argparse
    parser = argparse.ArgumentParser(description='IDS Detector - Voting Ensemble')
    parser.add_argument('--threshold', type=float, default=0.49,
                       help='Detection threshold (default: 0.49)')
    parser.add_argument('--file', '-f', type=str,
                       help='CSV file to analyze')
    parser.add_argument('--output', '-o', type=str,
                       help='Output file for results')
    parser.add_argument('--interactive', '-i', action='store_true',
                       help='Run in interactive mode')
    parser.add_argument('--model-dir', type=str, default='models/UNSW',
                       help='Directory containing models')
    
    args = parser.parse_args()
    
    # Initialize detector
    detector = IDSDetector(model_dir=args.model_dir, threshold=args.threshold)
    
    if args.interactive:
        detector.interactive_mode()
    elif args.file:
        detector.predict_file(args.file, args.output)
    else:
        print("\n📋 Usage examples:")
        print("  python deploy_ids.py --interactive")
        print("  python deploy_ids.py --file traffic.csv --output results.csv")
        print("  python deploy_ids.py --threshold 0.45 --interactive")


if __name__ == "__main__":
    main()